## Phase 3 - Feature Pipeline

This notebook builds the final supervised-learning table from the 1-second interim telemetry table.

Phase 3 goals:
1. Build past-only lagged and rolling features within each subject.
2. Define the regression target (`hr_target_30s`) and classification target (`activity_target`).
3. Drop rows made invalid by windowing and future target shift.
4. Save the final modeling table and Phase 3 validation artifacts.

In [2]:
from pathlib import Path

import pandas as pd

# Keep paths explicit so saves are reproducible from any working directory.
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INTERIM_PATH = REPO_ROOT / "data" / "interim" / "pamap2_per_second.parquet"
PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "pamap2_model_table.parquet"
METRICS_DIR = REPO_ROOT / "artifacts" / "metrics"

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Interim path exists: {INTERIM_PATH.exists()}")

Repo root: G:\Other computers\My Mac\Documents\pamap2_telemetry\pamap2_telemetry
Interim path exists: True


In [3]:
if not INTERIM_PATH.exists():
    raise FileNotFoundError(f"Missing interim table: {INTERIM_PATH}")

interim_df = pd.read_parquet(INTERIM_PATH).copy()

expected_columns = [
    "subject_id",
    "session",
    "timestamp_s",
    "activity_id",
    "activity_label",
    "heart_rate_bpm",
    "hand_acc_16g_mag",
    "chest_acc_16g_mag",
    "ankle_acc_16g_mag",
    "hand_gyro_mag",
    "chest_gyro_mag",
    "ankle_gyro_mag",
]

missing_columns = [c for c in expected_columns if c not in interim_df.columns]
extra_columns = [c for c in interim_df.columns if c not in expected_columns]
if missing_columns:
    raise ValueError(f"Interim table is missing required columns: {missing_columns}")
if extra_columns:
    print(f"Warning: interim table has extra columns that will be kept: {extra_columns}")

# Enforce deterministic order before any temporal feature logic.
model_df = interim_df.sort_values(["subject_id", "timestamp_s"]).reset_index(drop=True)

duplicate_rows = model_df.duplicated(subset=["subject_id", "timestamp_s"]).sum()
if duplicate_rows:
    raise ValueError(f"Found duplicate subject-second rows: {duplicate_rows}")

sorted_ok = (
    model_df.groupby("subject_id", sort=False)["timestamp_s"]
    .apply(lambda s: s.is_monotonic_increasing)
    .all()
)
if not sorted_ok:
    raise ValueError("Timestamps are not sorted within one or more subjects.")

print(f"Loaded rows: {len(model_df):,}")
print(f"Loaded columns: {len(model_df.columns)}")
display(model_df.head())

Loaded rows: 18,939
Loaded columns: 12


,subject_id,session,timestamp_s,activity_id,activity_label,heart_rate_bpm,hand_acc_16g_mag,chest_acc_16g_mag,ankle_acc_16g_mag,hand_gyro_mag,chest_gyro_mag,ankle_gyro_mag
0,101,protocol,37,1,lying,100.000000,9.806062,9.817320,9.866643,0.359794,0.059192,0.033850
1,101,protocol,38,1,lying,100.888889,9.805869,9.809444,9.866094,0.844113,0.127856,0.034384
2,101,protocol,39,1,lying,101.111111,9.765082,9.822854,9.857957,0.957831,0.420435,0.090013
3,101,protocol,40,1,lying,102.000000,10.086760,9.817623,9.845210,0.747128,0.110556,0.051834
4,101,protocol,41,1,lying,102.000000,9.794762,9.853255,9.847049,1.019548,0.090597,0.042753


### Build Past-Only Features

All lagged and rolling features are computed within each subject to prevent leakage across subject boundaries.

In [ ]:
signal_columns = [
    "heart_rate_bpm",
    "hand_acc_16g_mag",
    "chest_acc_16g_mag",
    "ankle_acc_16g_mag",
    "hand_gyro_mag",
    "chest_gyro_mag",
    "ankle_gyro_mag",
]

for col in signal_columns:
    grouped = model_df.groupby("subject_id", sort=False)[col]
    model_df[f"{col}_lag_1"] = grouped.shift(1)
    model_df[f"{col}_lag_5"] = grouped.shift(5)

    # min_periods enforces full windows so early rows are marked invalid and dropped later.
    roll_5 = grouped.transform(lambda s: s.rolling(window=5, min_periods=5).mean())
    roll_5_std = grouped.transform(lambda s: s.rolling(window=5, min_periods=5).std())
    roll_10 = grouped.transform(lambda s: s.rolling(window=10, min_periods=10).mean())
    roll_10_std = grouped.transform(lambda s: s.rolling(window=10, min_periods=10).std())

    model_df[f"{col}_rollmean_5"] = roll_5
    model_df[f"{col}_rollstd_5"] = roll_5_std
    model_df[f"{col}_rollmean_10"] = roll_10
    model_df[f"{col}_rollstd_10"] = roll_10_std
    model_df[f"{col}_delta_from_rollmean_5"] = model_df[col] - roll_5

feature_columns = []
for col in signal_columns:
    feature_columns.extend(
        [
            f"{col}_lag_1",
            f"{col}_lag_5",
            f"{col}_rollmean_5",
            f"{col}_rollstd_5",
            f"{col}_rollmean_10",
            f"{col}_rollstd_10",
            f"{col}_delta_from_rollmean_5",
        ]
    )

print(f"Created derived feature columns: {len(feature_columns)}")

AttributeError: module 'numpy' has no attribute 'matrix'

: 

### Create Targets And Drop Invalid Rows

In [ ]:
row_counts_before = (
    model_df.groupby("subject_id")
    .agg(rows_before=("subject_id", "size"))
    .reset_index()
)

# Regression target is heart rate 30 seconds ahead within each subject.
model_df["hr_target_30s"] = (
    model_df.groupby("subject_id", sort=False)["heart_rate_bpm"].shift(-30)
)

# Classification target is the current activity label at time t.
model_df["activity_target"] = model_df["activity_id"]

essential_columns = feature_columns + ["hr_target_30s", "activity_target"]

model_df_clean = model_df.dropna(subset=essential_columns).reset_index(drop=True)

row_counts_after = (
    model_df_clean.groupby("subject_id")
    .agg(rows_after=("subject_id", "size"))
    .reset_index()
)

row_retention_df = row_counts_before.merge(row_counts_after, on="subject_id", how="left")
row_retention_df["rows_after"] = row_retention_df["rows_after"].fillna(0).astype(int)
row_retention_df["rows_dropped"] = row_retention_df["rows_before"] - row_retention_df["rows_after"]
row_retention_df["retained_fraction"] = row_retention_df["rows_after"] / row_retention_df["rows_before"]

missing_rows = model_df_clean[essential_columns].isna().sum()
feature_missingness_df = pd.DataFrame(
    {
        "column_name": missing_rows.index,
        "missing_rows": missing_rows.values,
    }
)
feature_missingness_df["total_rows"] = len(model_df_clean)
feature_missingness_df["missing_fraction"] = (
    feature_missingness_df["missing_rows"] / feature_missingness_df["total_rows"]
)
feature_missingness_df = feature_missingness_df.sort_values(
    ["missing_rows", "column_name"], ascending=[False, True]
).reset_index(drop=True)

hr_target_stats = model_df_clean["hr_target_30s"].describe().reset_index()
hr_target_stats.columns = ["metric", "value"]
hr_target_stats.insert(0, "table", "hr_target_30s_summary")

activity_target_counts = (
    model_df_clean.groupby(["activity_target", "activity_label"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)
activity_target_counts.insert(0, "table", "activity_target_counts")

target_summary_df = pd.concat([hr_target_stats, activity_target_counts], ignore_index=True)

print(f"Rows before cleaning: {len(model_df):,}")
print(f"Rows after cleaning:  {len(model_df_clean):,}")
print(f"Rows dropped:         {len(model_df) - len(model_df_clean):,}")
display(row_retention_df)

### Save Final Outputs And Validation Artifacts

In [ ]:
model_df_clean.to_parquet(PROCESSED_PATH, index=False)

row_retention_path = METRICS_DIR / "phase3_row_retention_by_subject.csv"
feature_missingness_path = METRICS_DIR / "phase3_feature_missingness_post_clean.csv"
target_summary_path = METRICS_DIR / "phase3_target_summary.csv"

row_retention_df.to_csv(row_retention_path, index=False)
feature_missingness_df.to_csv(feature_missingness_path, index=False)
target_summary_df.to_csv(target_summary_path, index=False)

print(f"Saved final modeling table: {PROCESSED_PATH}")
print(f"Saved artifact: {row_retention_path}")
print(f"Saved artifact: {feature_missingness_path}")
print(f"Saved artifact: {target_summary_path}")

In [ ]:
print("Final table preview:")
display(model_df_clean.head())

print("\nFinal schema check:")
print(f"Rows: {len(model_df_clean):,}")
print(f"Columns: {len(model_df_clean.columns)}")
print("No duplicate subject-second rows:", not model_df_clean.duplicated(["subject_id", "timestamp_s"]).any())
print("No missing hr_target_30s:", model_df_clean["hr_target_30s"].isna().sum() == 0)
print("No missing activity_target:", model_df_clean["activity_target"].isna().sum() == 0)

display(feature_missingness_df.head(10))
display(target_summary_df.head(20))

## Phase 3 Completion Summary

Phase 3 is complete when:
1. `data/processed/pamap2_model_table.parquet` is generated.
2. `hr_target_30s` and `activity_target` are present with no missing values after cleaning.
3. Lagged and rolling features are created from subject-local past-only telemetry.
4. Validation artifacts are saved to `artifacts/metrics/phase3_*.csv`.